# E-Commerce Analytics System (WEEK-8 ASSIGNMENT)

We will do 5 things:
1. Create 4 CSV files with some messy data on purpose
2. Clean the messy data
3. Load the clean data into SQLite and run some SQL queries
4. Make a small tool that prints a summary report
5. Write simple tests for tricky/edge cases

## Step 0: Importing the libraries

In [ ]:
import random
import sqlite3
from datetime import datetime, timedelta

import pandas as pd

random.seed(42)

print("Libraries loaded!")


Libraries loaded!


## Part 1: Create the data

On purpose, we make some rows "dirty":
- Some orders have a missing `customer_id`
- Some order dates are written in the wrong format (`DD-MM-YYYY` instead of `YYYY-MM-DD`)
- Some order_items have a negative quantity (these represent returns)
- Some product names have extra spaces or weird capitalization
- Some customer emails are invalid


In [ ]:
first_names = ["Amit", "Priya", "Rahul", "Sneha", "Vikram", "Anjali", "Karan", "Neha",
               "Rohan", "Pooja", "Arjun", "Divya", "Manish", "Kavya", "Suresh", "Isha"]
last_names = ["Sharma", "Verma", "Patel", "Gupta", "Nair", "Reddy", "Iyer", "Singh",
              "Mehta", "Joshi", "Kapoor", "Das", "Chawla", "Rao", "Bose", "Malhotra"]

product_words = ["Smart", "Classic", "Pro", "Mini", "Ultra", "Basic", "Prime", "Eco",
                  "Deluxe", "Superb", "Comfort", "Trendy", "Royal", "Urban", "Cozy"]

categories = {
    "Electronics": ["Mobiles", "Laptops", "Audio", "Cameras"],
    "Clothing": ["Men", "Women", "Kids", "Footwear"],
    "Home": ["Furniture", "Kitchen", "Decor", "Lighting"],
    "Books": ["Fiction", "Non-Fiction", "Academic", "Comics"],
}

statuses = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
customer_types = ["REGULAR", "PREMIUM", "VIP"]
regions = ["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]

NUM_CUSTOMERS = 500
NUM_PRODUCTS = 520
NUM_ORDERS = 1200
NUM_ORDER_ITEMS = 2600

print("Setup done. Ready to generate data.")


Setup done. Ready to generate data.


In [3]:
# a small helper function to make a random date and time between two dates
def random_date(start, end):
    total_seconds = int((end - start).total_seconds())
    random_seconds = random.randint(0, total_seconds)
    return start + timedelta(seconds=random_seconds)

start_date = datetime(2023, 1, 1)
end_date = datetime(2025, 6, 30)


In [4]:
# ---------- 1. customers.csv ----------
customer_rows = []

for i in range(1, NUM_CUSTOMERS + 1):
    first = random.choice(first_names)
    last = random.choice(last_names)
    full_name = first + " " + last

    email = first.lower() + "." + last.lower() + str(i) + "@example.com"

    # make 2% of emails invalid on purpose
    if random.random() < 0.02:
        if random.random() < 0.5:
            email = email.replace("@", "")       # missing @
        else:
            email = first.lower() + "@"           # missing domain

    reg_date = random_date(start_date, end_date)

    row = {
        "customer_id": i,
        "customer_name": full_name,
        "email": email,
        "registration_date": reg_date.strftime("%Y-%m-%d %H:%M:%S"),
        "customer_type": random.choice(customer_types),
    }
    customer_rows.append(row)

customers = pd.DataFrame(customer_rows)
customers.head()


,customer_id,customer_name,email,registration_date,customer_type
0,1,Sneha Sharma,sneha.sharma1@example.com,2024-01-16 10:13:48,REGULAR
1,2,Vikram Gupta,vikram.gupta2@example.com,2025-04-27 04:44:17,REGULAR
2,3,Kavya Verma,kavya.verma3@example.com,2023-12-06 15:24:52,REGULAR
3,4,Amit Iyer,amit.iyer4@example.com,2025-04-26 12:53:27,PREMIUM
4,5,Neha Bose,neha.bose5@example.com,2023-01-11 02:17:28,REGULAR


In [5]:
# ---------- 2. products.csv ----------
product_rows = []

for i in range(1, NUM_PRODUCTS + 1):
    category = random.choice(list(categories.keys()))
    subcategory = random.choice(categories[category])
    name = random.choice(product_words) + " " + subcategory + " " + random.choice(product_words)

    # make about 15% of product names messy (extra spaces or wrong case) on purpose
    if random.random() < 0.15:
        messy_choice = random.choice(["upper", "lower", "spaces"])
        if messy_choice == "upper":
            name = name.upper()
        elif messy_choice == "lower":
            name = name.lower()
        else:
            name = "  " + name + "  "   # extra spaces at start/end

    row = {
        "product_id": i,
        "product_name": name,
        "category": category,
        "subcategory": subcategory,
        "cost_price": round(random.uniform(5, 2000), 2),
    }
    product_rows.append(row)

products = pd.DataFrame(product_rows)
products.head()


,product_id,product_name,category,subcategory,cost_price
0,1,Trendy Cameras Mini,Electronics,Cameras,586.21
1,2,comfort cameras eco,Electronics,Cameras,487.83
2,3,Royal Footwear Prime,Clothing,Footwear,961.07
3,4,Deluxe Comics Smart,Books,Comics,722.26
4,5,Royal Mobiles Mini,Electronics,Mobiles,713.24


In [6]:
# ---------- 3. orders.csv ----------
order_rows = []
customer_id_list = customers["customer_id"].tolist()

for i in range(1, NUM_ORDERS + 1):
    cust_id = random.choice(customer_id_list)

    # make 5% of customer_id missing on purpose
    if random.random() < 0.05:
        cust_id = ""   # empty means "missing"

    order_dt = random_date(start_date, end_date)

    # make about 10% of dates use the WRONG format (DD-MM-YYYY) on purpose
    if random.random() < 0.10:
        date_text = order_dt.strftime("%d-%m-%Y %H:%M:%S")
    else:
        date_text = order_dt.strftime("%Y-%m-%d %H:%M:%S")

    row = {
        "order_id": i,
        "customer_id": cust_id,
        "order_date": date_text,
        "status": random.choice(statuses),
        "region_code": random.choice(regions),
    }
    order_rows.append(row)

orders = pd.DataFrame(order_rows)
orders.head()


,order_id,customer_id,order_date,status,region_code
0,1,35,2025-02-26 02:45:30,PLACED,NORTH
1,2,352,2024-09-07 05:47:20,RETURNED,NORTH
2,3,427,2024-05-28 10:29:09,CANCELLED,WEST
3,4,,2023-07-30 21:16:19,CANCELLED,CENTRAL
4,5,459,2024-10-05 01:17:19,PLACED,CENTRAL


In [7]:
# ---------- 4. order_items.csv ----------
order_item_rows = []
order_id_list = orders["order_id"].tolist()
product_id_list = products["product_id"].tolist()
max_order_id = max(order_id_list)

for i in range(1, NUM_ORDER_ITEMS + 1):
    order_id = random.choice(order_id_list)

    # make about 1% of rows point to an order_id that does NOT exist, on purpose
    if random.random() < 0.01:
        order_id = max_order_id + random.randint(1, 500)

    quantity = random.randint(1, 6)

    # make 3% of rows have a negative quantity (these represent returns), on purpose
    if random.random() < 0.03:
        quantity = -random.randint(1, 3)

    row = {
        "item_id": i,
        "order_id": order_id,
        "product_id": random.choice(product_id_list),
        "quantity": quantity,
        "unit_price": round(random.uniform(5, 2500), 2),
        "discount_percent": round(random.uniform(0, 60), 1),
    }
    order_item_rows.append(row)

order_items = pd.DataFrame(order_item_rows)
order_items.head()


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,455,378,2,908.89,30.1
1,2,4,236,4,349.51,11.3
2,3,433,185,1,1067.15,17.1
3,4,793,432,4,452.90,23.9
4,5,1060,355,4,1716.73,6.1


In [ ]:
# save the raw (messy) data to CSV files.
import os
os.makedirs("data", exist_ok=True)

customers.to_csv("data/customers.csv", index=False)
products.to_csv("data/products.csv", index=False)
orders.to_csv("data/orders.csv", index=False)
order_items.to_csv("data/order_items.csv", index=False)

print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (500, 5)
products: (520, 5)
orders: (1200, 5)
order_items: (2600, 6)


## Part 2: Clean the data

Now we write simple functions to fix the problems we created on purpose.
Each function returns two things:
1. the cleaned data
2. a small dictionary that says how many problems it fixed (so we can print a report)


In [ ]:
def clean_orders(orders_df):
    """
    Fixes order_date format and handles missing customer_id.
    Returns: (cleaned dataframe, dictionary of issues found)
    """
    orders_df = orders_df.copy()

    fixed_dates = 0
    missing_customers = 0

    new_dates = []
    new_customers = []

    for index, row in orders_df.iterrows():
        date_text = str(row["order_date"]).strip()

        # try the correct format first
        fixed_date = None
        try:
            parsed = datetime.strptime(date_text, "%Y-%m-%d %H:%M:%S")
            fixed_date = parsed.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            # maybe it's the wrong format (DD-MM-YYYY), try that instead
            try:
                parsed = datetime.strptime(date_text, "%d-%m-%Y %H:%M:%S")
                fixed_date = parsed.strftime("%Y-%m-%d %H:%M:%S")
                fixed_dates = fixed_dates + 1
            except ValueError:
                fixed_date = None   # could not understand this date at all

        new_dates.append(fixed_date)

        # check if customer_id is missing
        cust_id = row["customer_id"]
        if pd.isna(cust_id) or str(cust_id).strip() in ["", "NULL", "None", "nan"]:
            new_customers.append(None)
            missing_customers = missing_customers + 1
        else:
            new_customers.append(int(cust_id))

    orders_df["order_date"] = new_dates
    orders_df["customer_id"] = new_customers

    issues = {
        "orders_date_format_fixed": fixed_dates,
        "orders_missing_customer_id": missing_customers,
    }
    return orders_df, issues


In [ ]:
def clean_products(products_df):
    """
    Removes extra spaces and makes product names use Title Case.
    Returns: (cleaned dataframe, dictionary of issues found)
    """
    products_df = products_df.copy()
    changed_count = 0

    new_names = []
    for index, row in products_df.iterrows():
        old_name = str(row["product_name"])

        # remove leading/trailing spaces, then collapse double spaces into one
        clean_name = " ".join(old_name.split())
        clean_name = clean_name.title()

        if clean_name != old_name:
            changed_count = changed_count + 1

        new_names.append(clean_name)

    products_df["product_name"] = new_names

    issues = {"products_names_cleaned": changed_count}
    return products_df, issues


In [ ]:
def validate_emails(customers_df):
    """
    Checks every email in a simple way: it must contain "@" and a "." after the "@".
    Returns: (list of customer_ids with bad emails, dictionary of issue count)
    """
    bad_customer_ids = []

    for index, row in customers_df.iterrows():
        email = str(row["email"])

        is_valid = True
        if "@" not in email:
            is_valid = False
        else:
            # check there is something before @ and a "." after @
            before_at, after_at = email.split("@", 1)
            if before_at == "" or "." not in after_at or after_at == "":
                is_valid = False

        if not is_valid:
            bad_customer_ids.append(row["customer_id"])

    issues = {"customers_invalid_emails": len(bad_customer_ids)}
    return bad_customer_ids, issues


In [12]:
def check_referential_integrity(orders_df, order_items_df):
    """
    Finds order_items rows whose order_id does not exist in the orders table.
    Returns: (dataframe of bad rows, dictionary of issue count)
    """
    valid_order_ids = set(orders_df["order_id"].tolist())

    bad_rows = []
    for index, row in order_items_df.iterrows():
        if row["order_id"] not in valid_order_ids:
            bad_rows.append(row)

    bad_df = pd.DataFrame(bad_rows)

    issues = {"order_items_orphan_rows": len(bad_df)}
    return bad_df, issues


In [ ]:
cleaned_orders, issues_1 = clean_orders(orders)
cleaned_products, issues_2 = clean_products(products)
bad_email_ids, issues_3 = validate_emails(customers)
orphan_items, issues_4 = check_referential_integrity(orders, order_items)

report = {}
report.update(issues_1)
report.update(issues_2)
report.update(issues_3)
report.update(issues_4)

print("=== DATA QUALITY REPORT ===")
for key in report:
    print(key, ":", report[key])

=== DATA QUALITY REPORT ===
orders_date_format_fixed : 111
orders_missing_customer_id : 66
products_names_cleaned : 96
customers_invalid_emails : 8
order_items_orphan_rows : 32


In [ ]:
# remove the "orphan" order_items rows (the ones pointing to a fake order_id)
if len(orphan_items) > 0:
    bad_ids = orphan_items["item_id"].tolist()
    cleaned_order_items = order_items[~order_items["item_id"].isin(bad_ids)].copy()
else:
    cleaned_order_items = order_items.copy()

print("order_items before cleaning:", len(order_items))
print("order_items after removing orphan rows:", len(cleaned_order_items))


order_items before cleaning: 2600
order_items after removing orphan rows: 2568


In [15]:
# save the cleaned files
os.makedirs("data/cleaned", exist_ok=True)

cleaned_orders.to_csv("data/cleaned/orders.csv", index=False)
cleaned_products.to_csv("data/cleaned/products.csv", index=False)
cleaned_order_items.to_csv("data/cleaned/order_items.csv", index=False)
customers.to_csv("data/cleaned/customers.csv", index=False)

with open("data/cleaned/data_quality_report.txt", "w") as f:
    f.write("DATA QUALITY REPORT\n")
    for key in report:
        f.write(key + ": " + str(report[key]) + "\n")

print("Cleaned files saved!")


Cleaned files saved!


## Part 3: SQL Analysis

Now we put the cleaned data into a small SQLite database (SQLite just stores everything
inside one file/memory, no server needed) and run SQL queries on it, from simple to advanced.


In [ ]:
conn = sqlite3.connect(":memory:")

cleaned_orders.to_sql("orders", conn, index=False, if_exists="replace")
cleaned_products.to_sql("products", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
cleaned_order_items.to_sql("order_items", conn, index=False, if_exists="replace")

print("Tables loaded into SQLite!")


Tables loaded into SQLite!


### Query 1: Total revenue per category
`revenue = quantity x unit_price x (1 - discount_percent/100)`

In [17]:
query1 = """
SELECT p.category,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""
pd.read_sql(query1, conn)


,category,total_revenue
0,Electronics,1914801.24
1,Home,1873311.76
2,Clothing,1823266.03
3,Books,1736035.38


### Query 2: Top 10 customers by total order value

In [18]:
query2 = """
SELECT o.customer_id,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.customer_id IS NOT NULL
GROUP BY o.customer_id
ORDER BY total_value DESC
LIMIT 10;
"""
pd.read_sql(query2, conn)


,customer_id,total_value
0,68.0,94093.36
1,34.0,78183.25
2,296.0,58182.37
3,487.0,56159.66
4,422.0,51273.98
5,129.0,49727.92
6,216.0,48169.07
7,405.0,46743.66
8,155.0,46033.40
9,228.0,44724.58


### Query 3: Number of orders per month (last 12 months)

In [19]:
query3 = """
SELECT strftime('%Y-%m', order_date) AS year_month,
       COUNT(*) AS order_count
FROM orders
WHERE order_date >= date((SELECT MAX(order_date) FROM orders), '-12 months')
GROUP BY year_month
ORDER BY year_month;
"""
pd.read_sql(query3, conn)


,year_month,order_count
0,2024-06,2
1,2024-07,39
2,2024-08,55
3,2024-09,48
4,2024-10,37
5,2024-11,36
6,2024-12,46
7,2025-01,48
8,2025-02,31
9,2025-03,46


### Query 4: Customers who ordered but never got anything delivered

In [20]:
query4 = """
SELECT DISTINCT customer_id
FROM orders
WHERE customer_id IS NOT NULL
AND customer_id NOT IN (
    SELECT customer_id FROM orders WHERE status = 'DELIVERED' AND customer_id IS NOT NULL
);
"""
pd.read_sql(query4, conn).head(10)


,customer_id
0,35.0
1,352.0
2,427.0
3,154.0
4,296.0
5,58.0
6,3.0
7,81.0
8,483.0
9,160.0


### Query 5: Products that were returned more than they were bought

In [21]:
query5 = """
SELECT oi.product_id,
       p.product_name,
       SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased_qty,
       SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_qty
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY oi.product_id, p.product_name
HAVING returned_qty > purchased_qty;
"""
pd.read_sql(query5, conn)


,product_id,product_name,purchased_qty,returned_qty
0,260,Cozy Non-Fiction Comfort,1,2
1,464,Ultra Men Mini,1,2


### Query 6: Return rate per category (returned items / total items)

In [22]:
query6 = """
SELECT p.category,
       SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_items,
       SUM(ABS(oi.quantity)) AS total_items,
       ROUND(1.0 * SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) / SUM(ABS(oi.quantity)), 4) AS return_rate
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY return_rate DESC;
"""
pd.read_sql(query6, conn)


,category,returned_items,total_items,return_rate
0,Electronics,65,2368,0.0274
1,Home,62,2322,0.0267
2,Clothing,40,2179,0.0184
3,Books,36,2114,0.0170


### Query 7: Running total of revenue per region (window function)

In [23]:
query7 = """
WITH daily AS (
    SELECT o.region_code,
           date(o.order_date) AS order_date,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.region_code, date(o.order_date)
)
SELECT region_code, order_date,
       ROUND(daily_revenue, 2) AS daily_revenue,
       ROUND(SUM(daily_revenue) OVER (PARTITION BY region_code ORDER BY order_date), 2) AS running_total
FROM daily
ORDER BY region_code, order_date;
"""
pd.read_sql(query7, conn).head(15)


,region_code,order_date,daily_revenue,running_total
0,CENTRAL,2023-01-06,2238.40,2238.40
1,CENTRAL,2023-01-07,7343.65,9582.05
2,CENTRAL,2023-01-15,7521.84,17103.88
3,CENTRAL,2023-01-16,8235.66,25339.54
4,CENTRAL,2023-01-17,2046.33,27385.87
5,CENTRAL,2023-01-20,9012.92,36398.79
6,CENTRAL,2023-01-22,6713.66,43112.45
7,CENTRAL,2023-01-28,7195.17,50307.62
8,CENTRAL,2023-02-02,9449.99,59757.61
9,CENTRAL,2023-02-04,8694.39,68452.00


### Query 8: Rank products inside each category by revenue

In [24]:
query8 = """
WITH revenue_per_product AS (
    SELECT p.category, p.product_name,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT category, product_name, ROUND(total_revenue, 2) AS total_revenue,
       DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM revenue_per_product
ORDER BY category, rank_in_category;
"""
pd.read_sql(query8, conn).head(15)


,category,product_name,total_revenue,rank_in_category
0,Books,Urban Non-Fiction Comfort,48529.62,1
1,Books,Eco Non-Fiction Basic,44515.74,2
2,Books,Royal Academic Deluxe,36847.74,3
3,Books,Eco Academic Trendy,35733.17,4
4,Books,Prime Fiction Pro,35516.63,5
5,Books,Pro Academic Eco,34643.62,6
6,Books,Smart Comics Prime,33635.78,7
7,Books,Mini Non-Fiction Trendy,33469.03,8
8,Books,Royal Academic Mini,32880.12,9
9,Books,Prime Comics Prime,31582.71,10


### Query 9: Days between a customer's orders (LAG), and flag "At Risk" customers

In [25]:
query9 = """
WITH with_previous AS (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
)
SELECT customer_id, order_date, previous_order_date,
       ROUND(julianday(order_date) - julianday(previous_order_date), 1) AS days_gap
FROM with_previous
WHERE previous_order_date IS NOT NULL
ORDER BY customer_id, order_date;
"""
pd.read_sql(query9, conn).head(10)


,customer_id,order_date,previous_order_date,days_gap
0,2.0,2023-03-22 13:35:06,2023-01-27 15:34:18,53.9
1,2.0,2025-02-01 05:51:21,2023-03-22 13:35:06,681.7
2,3.0,2023-10-17 04:27:44,2023-03-10 09:33:21,220.8
3,3.0,2024-06-19 11:47:23,2023-10-17 04:27:44,246.3
4,3.0,2024-10-05 23:47:33,2024-06-19 11:47:23,108.5
5,4.0,2025-02-02 20:46:19,2024-09-07 07:09:47,148.6
6,6.0,2023-08-26 18:03:57,2023-01-25 09:37:45,213.4
7,6.0,2024-04-25 04:49:59,2023-08-26 18:03:57,242.4
8,6.0,2024-05-17 03:29:00,2024-04-25 04:49:59,21.9
9,6.0,2024-12-22 02:58:09,2024-05-17 03:29:00,219.0


In [26]:
query9b = """
WITH with_previous AS (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
),
gaps AS (
    SELECT customer_id,
           julianday(order_date) - julianday(previous_order_date) AS days_gap
    FROM with_previous
    WHERE previous_order_date IS NOT NULL
)
SELECT customer_id,
       ROUND(AVG(days_gap), 1) AS average_gap_days,
       CASE WHEN AVG(days_gap) > 30 THEN 'At Risk' ELSE 'Healthy' END AS status
FROM gaps
GROUP BY customer_id
ORDER BY average_gap_days DESC;
"""
pd.read_sql(query9b, conn).head(10)


,customer_id,average_gap_days,status
0,251.0,799.7,At Risk
1,183.0,782.4,At Risk
2,221.0,761.8,At Risk
3,58.0,748.6,At Risk
4,438.0,735.6,At Risk
5,263.0,730.7,At Risk
6,109.0,726.4,At Risk
7,41.0,722.6,At Risk
8,225.0,698.8,At Risk
9,435.0,692.2,At Risk


### Query 10: Monthly revenue per customer, grouped into High/Medium/Low tiers

In [27]:
query10 = """
WITH monthly_revenue AS (
    SELECT o.customer_id,
           strftime('%Y-%m', o.order_date) AS year_month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id, year_month
),
tagged AS (
    SELECT customer_id, year_month, revenue,
           CASE
               WHEN revenue > 10000 THEN 'High'
               WHEN revenue >= 5000 THEN 'Medium'
               ELSE 'Low'
           END AS revenue_tier
    FROM monthly_revenue
)
SELECT year_month, revenue_tier, COUNT(DISTINCT customer_id) AS customer_count
FROM tagged
GROUP BY year_month, revenue_tier
ORDER BY year_month, revenue_tier;
"""
pd.read_sql(query10, conn).head(15)


,year_month,revenue_tier,customer_count
0,2023-01,High,5
1,2023-01,Low,6
2,2023-01,Medium,19
3,2023-02,High,7
4,2023-02,Low,12
5,2023-02,Medium,3
6,2023-03,High,9
7,2023-03,Low,14
8,2023-03,Medium,6
9,2023-04,High,5


### Query 11: Split customers into 4 groups (quartiles) by total spend

In [28]:
query11 = """
WITH customer_value AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT customer_id, ROUND(total_value, 2) AS total_value,
       NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
       CASE NTILE(4) OVER (ORDER BY total_value DESC)
            WHEN 1 THEN 'Platinum'
            WHEN 2 THEN 'Gold'
            WHEN 3 THEN 'Silver'
            ELSE 'Bronze'
       END AS quartile_label
FROM customer_value
ORDER BY total_value DESC;
"""
pd.read_sql(query11, conn).head(10)


,customer_id,total_value,quartile,quartile_label
0,68.0,94093.36,1,Platinum
1,34.0,78183.25,1,Platinum
2,296.0,58182.37,1,Platinum
3,487.0,56159.66,1,Platinum
4,422.0,51273.98,1,Platinum
5,129.0,49727.92,1,Platinum
6,216.0,48169.07,1,Platinum
7,405.0,46743.66,1,Platinum
8,155.0,46033.40,1,Platinum
9,228.0,44724.58,1,Platinum


### Query 12: Compare each month's revenue to the same month last year

In [29]:
query12 = """
WITH monthly AS (
    SELECT strftime('%Y', o.order_date) AS year,
           strftime('%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY year, month
)
SELECT curr.year, curr.month,
       ROUND(curr.revenue, 2) AS revenue,
       ROUND(prev.revenue, 2) AS prev_year_revenue,
       CASE WHEN prev.revenue IS NOT NULL AND prev.revenue != 0
            THEN ROUND(100.0 * (curr.revenue - prev.revenue) / prev.revenue, 2)
            ELSE NULL
       END AS yoy_growth_percent
FROM monthly curr
LEFT JOIN monthly prev
    ON prev.month = curr.month
   AND CAST(prev.year AS INTEGER) = CAST(curr.year AS INTEGER) - 1
ORDER BY curr.year, curr.month;
"""
pd.read_sql(query12, conn)


,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2023,01,234976.40,NaN,NaN
1,2023,02,193157.45,NaN,NaN
2,2023,03,243993.47,NaN,NaN
3,2023,04,147685.09,NaN,NaN
4,2023,05,322668.16,NaN,NaN
5,2023,06,274343.14,NaN,NaN
6,2023,07,359817.71,NaN,NaN
7,2023,08,277708.14,NaN,NaN
8,2023,09,203599.67,NaN,NaN
9,2023,10,199634.59,NaN,NaN


### Query 13: Each customer's first vs most recent purchase category

In [30]:
query13 = """
WITH customer_categories AS (
    SELECT o.customer_id, o.order_date, p.category,
           FIRST_VALUE(p.category) OVER (
               PARTITION BY o.customer_id ORDER BY o.order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
           ) AS first_category,
           LAST_VALUE(p.category) OVER (
               PARTITION BY o.customer_id ORDER BY o.order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
           ) AS last_category
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.customer_id IS NOT NULL
)
SELECT DISTINCT customer_id, first_category, last_category,
       CASE WHEN first_category != last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM customer_categories
ORDER BY customer_id;
"""
pd.read_sql(query13, conn).head(10)


,customer_id,first_category,last_category,category_shift
0,2.0,Home,Books,Yes
1,3.0,Electronics,Home,Yes
2,4.0,Electronics,Home,Yes
3,5.0,Books,Clothing,Yes
4,6.0,Home,Clothing,Yes
5,7.0,Clothing,Home,Yes
6,8.0,Clothing,Electronics,Yes
7,10.0,Home,Electronics,Yes
8,11.0,Books,Electronics,Yes
9,13.0,Electronics,Electronics,No


### Query 14: What % of total revenue comes from the top customers

In [31]:
query14 = """
WITH customer_revenue AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT customer_id,
       ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY revenue DESC), 2) AS cumulative_revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER (), 2) AS cumulative_percent
FROM customer_revenue
ORDER BY revenue DESC;
"""
pd.read_sql(query14, conn).head(10)


,customer_id,revenue,cumulative_revenue,cumulative_percent
0,68.0,94093.36,94093.36,1.37
1,34.0,78183.25,172276.61,2.51
2,296.0,58182.37,230458.98,3.35
3,487.0,56159.66,286618.64,4.17
4,422.0,51273.98,337892.62,4.92
5,129.0,49727.92,387620.54,5.64
6,216.0,48169.07,435789.61,6.34
7,405.0,46743.66,482533.27,7.02
8,155.0,46033.40,528566.68,7.69
9,228.0,44724.58,573291.26,8.34


### Query 15: Cohort analysis — how many customers keep ordering after they sign up

In [32]:
query15 = """
WITH cohort AS (
    SELECT customer_id, strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
customer_order_months AS (
    SELECT o.customer_id, strftime('%Y-%m', o.order_date) AS order_month
    FROM orders o
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id, order_month
),
months_since_signup AS (
    SELECT c.customer_id, c.cohort_month,
           (CAST(strftime('%Y', com.order_month || '-01') AS INTEGER) * 12 +
            CAST(strftime('%m', com.order_month || '-01') AS INTEGER)) -
           (CAST(strftime('%Y', c.cohort_month || '-01') AS INTEGER) * 12 +
            CAST(strftime('%m', c.cohort_month || '-01') AS INTEGER)) AS month_number
    FROM cohort c
    JOIN customer_order_months com ON com.customer_id = c.customer_id
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_customers
    FROM cohort
    GROUP BY cohort_month
)
SELECT m.cohort_month, m.month_number,
       COUNT(DISTINCT m.customer_id) AS active_customers,
       cs.cohort_customers,
       ROUND(100.0 * COUNT(DISTINCT m.customer_id) / cs.cohort_customers, 2) AS retention_rate
FROM months_since_signup m
JOIN cohort_size cs ON cs.cohort_month = m.cohort_month
WHERE m.month_number BETWEEN 0 AND 3
GROUP BY m.cohort_month, m.month_number
ORDER BY m.cohort_month, m.month_number;
"""
pd.read_sql(query15, conn).head(15)


,cohort_month,month_number,active_customers,cohort_customers,retention_rate
0,2023-01,0,1,16,6.25
1,2023-01,2,1,16,6.25
2,2023-02,0,1,9,11.11
3,2023-02,1,2,9,22.22
4,2023-02,3,1,9,11.11
5,2023-03,0,3,26,11.54
6,2023-03,2,3,26,11.54
7,2023-03,3,3,26,11.54
8,2023-04,0,2,16,12.50
9,2023-04,1,1,16,6.25


### Query 16: Which products are often bought together

In [33]:
query16 = """
SELECT a.product_id AS product_a, b.product_id AS product_b,
       COUNT(*) AS times_bought_together
FROM order_items a
JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id
ORDER BY times_bought_together DESC
LIMIT 20;
"""
pd.read_sql(query16, conn)


,product_a,product_b,times_bought_together
0,2,423,2
1,4,311,2
2,11,25,2
3,13,492,2
4,21,65,2
5,21,378,2
6,31,484,2
7,37,61,2
8,87,311,2
9,103,311,2


## Part 4: Summary report tool

This is a function that:
1. Takes a report type, a start date, and an end date
2. Looks up total orders, revenue, and unique customers for that period
3. Also finds the top 3 products
4. Compares the numbers to the *previous* period of the same length


In [34]:
def get_period_totals(conn, start_text, end_text):
    """Returns total orders, unique customers, and revenue between two datetime strings."""
    query = """
    SELECT COUNT(DISTINCT o.order_id) AS total_orders,
           COUNT(DISTINCT o.customer_id) AS unique_customers,
           COALESCE(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 0) AS revenue
    FROM orders o
    LEFT JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_date BETWEEN ? AND ?
    """
    result = pd.read_sql(query, conn, params=[start_text, end_text])
    return result.iloc[0]


def get_top_3_products(conn, start_text, end_text):
    query = """
    SELECT p.product_name,
           ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_date BETWEEN ? AND ?
    GROUP BY p.product_name
    ORDER BY revenue DESC
    LIMIT 3
    """
    return pd.read_sql(query, conn, params=[start_text, end_text])


def percent_change(old_value, new_value):
    if old_value == 0 or pd.isna(old_value):
        return None
    return round(100.0 * (new_value - old_value) / old_value, 2)


def generate_summary_report(conn, report_type, start_date_text, end_date_text):
    """
    report_type      : just a label, e.g. "daily", "weekly", "monthly"
    start_date_text  : "YYYY-MM-DD"
    end_date_text    : "YYYY-MM-DD"
    """
    start_dt = datetime.strptime(start_date_text, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date_text, "%Y-%m-%d")
    period_length = end_dt - start_dt

    # work out the previous period of the same length, right before this one
    previous_end = start_dt - timedelta(seconds=1)
    previous_start = previous_end - period_length

    current = get_period_totals(conn, start_dt.strftime("%Y-%m-%d %H:%M:%S"), end_dt.strftime("%Y-%m-%d %H:%M:%S"))
    previous = get_period_totals(conn, previous_start.strftime("%Y-%m-%d %H:%M:%S"), previous_end.strftime("%Y-%m-%d %H:%M:%S"))

    top_products = get_top_3_products(conn, start_dt.strftime("%Y-%m-%d %H:%M:%S"), end_dt.strftime("%Y-%m-%d %H:%M:%S"))

    print("=== " + report_type.upper() + " REPORT: " + start_date_text + " to " + end_date_text + " ===")
    print("Total Orders     :", int(current["total_orders"]))
    print("Revenue          :", round(current["revenue"], 2))
    print("Unique Customers :", int(current["unique_customers"]))

    print("Top 3 Products:")
    for index, row in top_products.iterrows():
        print("   -", row["product_name"], ":", row["revenue"])

    print("Compared to previous period:")
    print("   - Orders change    :", percent_change(previous["total_orders"], current["total_orders"]), "%")
    print("   - Revenue change   :", percent_change(previous["revenue"], current["revenue"]), "%")
    print("   - Customers change :", percent_change(previous["unique_customers"], current["unique_customers"]), "%")


In [ ]:
generate_summary_report(conn, "monthly", "2024-06-01", "2024-06-30")

=== MONTHLY REPORT: 2024-06-01 to 2024-06-30 ===
Total Orders     : 34
Revenue          : 225337.36
Unique Customers : 31
Top 3 Products:
   - Royal Lighting Eco : 12291.41
   - Mini Non-Fiction Trendy : 11863.51
   - Cozy Academic Basic : 10727.96
Compared to previous period:
   - Orders change    : -8.11 %
   - Revenue change   : -12.2 %
   - Customers change : -6.06 %


## Part 5: Testing tricky edge cases

In [ ]:
def test_order_item_points_to_missing_order():
    """What happens when order_items has an order_id that is not in orders?"""
    fake_orders = pd.DataFrame({"order_id": [1, 2]})
    fake_items = pd.DataFrame({"item_id": [10, 11], "order_id": [1, 99]})

    valid_ids = set(fake_orders["order_id"])
    bad_rows = fake_items[~fake_items["order_id"].isin(valid_ids)]

    if len(bad_rows) == 1 and bad_rows.iloc[0]["item_id"] == 11:
        print("PASS: the row with order_id = 99 was correctly found as an orphan row")
    else:
        print("FAIL: orphan row was not detected correctly")


def test_discount_over_100():
    """What happens when discount_percent is more than 100?"""
    quantity = 2
    unit_price = 100.0
    discount_percent = 150   # more than 100 not allowed

    revenue = quantity * unit_price * (1 - discount_percent / 100.0)

    if revenue < 0:
        print("PASS: discount_percent = 150 gives a negative revenue (", revenue, "), so it can be flagged as bad data")
    else:
        print("FAIL: expected a negative revenue for discount over 100%")


def test_zero_quantity():
    """What happens when quantity is 0?"""
    quantity = 0
    unit_price = 50.0
    discount_percent = 10

    revenue = quantity * unit_price * (1 - discount_percent / 100.0)

    if revenue == 0:
        print("PASS: quantity = 0 gives 0 revenue, so it does not count as a purchase or a return")
    else:
        print("FAIL: expected 0 revenue when quantity is 0")


def test_future_order_date():
    """What happens when order_date is in the future?"""
    future_date = datetime.now() + timedelta(days=30)
    today = datetime.now()

    if future_date > today:
        print("PASS: a date 30 days from now is correctly detected as being in the future")
    else:
        print("FAIL: future date was not detected correctly")


In [37]:
print("=== EDGE CASE TEST RESULTS ===")
test_order_item_points_to_missing_order()
test_discount_over_100()
test_zero_quantity()
test_future_order_date()


=== EDGE CASE TEST RESULTS ===
PASS: the row with order_id = 99 was correctly found as an orphan row
PASS: discount_percent = 150 gives a negative revenue ( -100.0 ), so it can be flagged as bad data
PASS: quantity = 0 gives 0 revenue, so it does not count as a purchase or a return
PASS: a date 30 days from now is correctly detected as being in the future
